In [ ]:
# ==========================================
# SYSTEM DEPENDENCIES
# ==========================================

!apt-get update -qq
!apt-get install -y -qq libgl1-mesa-dev libgl1-mesa-glx libglew-dev \
                         libosmesa6-dev software-properties-common patchelf

print("✅ System dependencies installed")

In [ ]:
# ==========================================
# ENVIRONMENT SETUP
# ==========================================

import os
import sys
from pathlib import Path

os.environ['MUJOCO_GL'] = 'egl'
os.environ['LEROBOT_VIDEO_BACKEND'] = 'pyav'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['SVT_LOG'] = '0'

# Get Kaggle secrets
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
    os.environ['WANDB_API_KEY'] = secrets.get_secret('WANDB_API_KEY')
    print("✅ Secrets loaded")
except:
    print("⚠️  Set secrets manually")

import wandb
wandb.login(key=os.environ.get('WANDB_API_KEY', ''))
print("✅ Environment configured")

In [ ]:
# ==========================================
# CLONE & INSTALL LEROBOT
# ==========================================

!git clone https://github.com/huggingface/lerobot.git /kaggle/working/lerobot
%cd /kaggle/working/lerobot
!pip install -e . -q
!pip install metaworld wandb opencv-python imageio imageio-ffmpeg av transformers -q

print("\n✅ All dependencies installed")

In [ ]:
# ==========================================
# ADD SRC TO PYTHON PATH
# ==========================================

import sys
from pathlib import Path

LEROBOT_DIR = Path("/kaggle/working/lerobot")
SRC_DIR = LEROBOT_DIR / "src"

if SRC_DIR.exists():
    sys.path.insert(0, str(SRC_DIR))
    print(f"✅ Added to Python path: {SRC_DIR}")

from lerobot.envs.metaworld import MetaworldEnv, TASK_DESCRIPTIONS
from lerobot.datasets.lerobot_dataset import LeRobotDataset
print("✅ LeRobot imports successful!")

---
## 📋 Configuration

**You can start with just 2 tasks!** Add more later.

In [ ]:
# ==========================================
# MULTI-TASK CONFIGURATION
# ==========================================

HF_USERNAME = "aryannzzz"

# Start with 2 tasks - add more as you generate datasets
TASKS = [
    "pick-place-v3",
    "handle-pull-v3",
    # Uncomment as you add more datasets:
    # "push-v3",
    # "reach-v3",
    # "shelf-place-v3",
    # "pick-place-wall-v3",
]

# Dataset repo IDs
DATASET_REPO_IDS = {
    task: f"{HF_USERNAME}/metaworld-{task}-expert" 
    for task in TASKS
}

# Training config
TRAINING_STEPS = 150000  # More steps for multi-task
BATCH_SIZE = 8
LEARNING_RATE = 0.0001
TASK_EMBED_DIM = 64  # Task embedding dimension

OUTPUT_DIR = "/kaggle/working/outputs/multitask_act"

print("📋 Multi-Task Configuration:")
print(f"   Tasks: {TASKS}")
print(f"   Datasets: {list(DATASET_REPO_IDS.values())}")
print(f"   Steps: {TRAINING_STEPS:,}")
print(f"   Task embed dim: {TASK_EMBED_DIM}")

---
## 🏗️ Task-Conditioned ACT Implementation

Based on LeX-O approach: Add task token embedding to ACT

In [ ]:
# ==========================================
# TASK-CONDITIONED ACT POLICY
# ==========================================

import torch
import torch.nn as nn
from transformers import AutoTokenizer
from lerobot.policies.act.modeling_act import ACTPolicy
from lerobot.policies.act.configuration_act import ACTConfig
from collections import deque
import copy

class TaskConditionedACTPolicy(nn.Module):
    """
    Task-Conditioned ACT Policy.
    
    Based on LeX-O architecture:
    - Adds a task token embedding to the ACT policy
    - Task name is encoded using a tokenizer
    - Task embedding is concatenated with observations
    
    Reference: https://github.com/aadarshram/lerobot/tree/LeX-O_TicTacToe
    """
    
    def __init__(
        self,
        base_config: dict,
        task_names: list[str],
        task_embed_dim: int = 64,
        tokenizer_name: str = "bert-base-uncased",
    ):
        super().__init__()
        
        self.task_names = task_names
        self.task_embed_dim = task_embed_dim
        self.num_tasks = len(task_names)
        
        # Task name to index mapping
        self.task_to_idx = {task: i for i, task in enumerate(task_names)}
        
        # Tokenizer for task names (like LeX-O)
        print(f"📥 Loading tokenizer: {tokenizer_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
        
        # Task embedding layer
        # Option 1: Simple learnable embedding per task
        self.task_embedding = nn.Embedding(self.num_tasks, task_embed_dim)
        
        # Option 2: Token-based embedding (like LeX-O)
        # This projects tokenized task name to embedding
        vocab_size = self.tokenizer.vocab_size
        self.token_embedding = nn.Embedding(vocab_size, task_embed_dim)
        self.task_projector = nn.Sequential(
            nn.Linear(task_embed_dim, task_embed_dim),
            nn.ReLU(),
            nn.Linear(task_embed_dim, task_embed_dim),
        )
        
        # Base ACT config - modify to accept task embedding
        self.base_config = copy.deepcopy(base_config)
        
        # Increase state dim to include task embedding
        original_state_dim = base_config.get('input_shapes', {}).get('observation.state', [4])[0]
        self.original_state_dim = original_state_dim
        self.augmented_state_dim = original_state_dim + task_embed_dim
        
        # Update config for augmented state
        modified_config = copy.deepcopy(base_config)
        if 'input_shapes' in modified_config:
            modified_config['input_shapes']['observation.state'] = [self.augmented_state_dim]
        
        # Create base ACT policy with modified config
        print(f"🔧 Creating ACT with augmented state dim: {self.augmented_state_dim}")
        self.act_config = ACTConfig(**modified_config)
        self.act_policy = ACTPolicy(self.act_config)
        
        # Current task for inference
        self._current_task = None
        self._current_task_embedding = None
        
        # Action queue for temporal ensemble
        self._action_queue = None
        
        print(f"✅ TaskConditionedACT initialized")
        print(f"   Tasks: {task_names}")
        print(f"   Task embed dim: {task_embed_dim}")
    
    def get_task_embedding_simple(self, task_name: str) -> torch.Tensor:
        """Get task embedding using simple lookup."""
        task_idx = self.task_to_idx[task_name]
        task_idx_tensor = torch.tensor([task_idx], device=self.task_embedding.weight.device)
        return self.task_embedding(task_idx_tensor)  # [1, embed_dim]
    
    def get_task_embedding_tokenized(self, task_name: str) -> torch.Tensor:
        """
        Get task embedding using tokenizer (LeX-O style).
        
        Tokenize the task name, embed tokens, and pool to single embedding.
        """
        # Tokenize task name
        tokens = self.tokenizer(
            task_name,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=16,
        )
        input_ids = tokens['input_ids'].to(self.token_embedding.weight.device)
        
        # Embed tokens
        token_embeds = self.token_embedding(input_ids)  # [1, seq_len, embed_dim]
        
        # Mean pooling over tokens
        task_embed = token_embeds.mean(dim=1)  # [1, embed_dim]
        
        # Project
        task_embed = self.task_projector(task_embed)
        
        return task_embed
    
    def set_task(self, task_name: str, use_tokenized: bool = False):
        """
        Set current task for inference.
        
        Args:
            task_name: Name of the task
            use_tokenized: Use tokenizer-based embedding (True) or simple lookup (False)
        """
        if task_name not in self.task_to_idx:
            raise ValueError(f"Unknown task: {task_name}. Known tasks: {self.task_names}")
        
        self._current_task = task_name
        
        if use_tokenized:
            self._current_task_embedding = self.get_task_embedding_tokenized(task_name)
        else:
            self._current_task_embedding = self.get_task_embedding_simple(task_name)
        
        print(f"🎯 Task set to: {task_name}")
    
    def reset(self):
        """Reset action queue for new episode."""
        self._action_queue = None
        if hasattr(self.act_policy, 'reset'):
            self.act_policy.reset()
    
    def augment_observation(self, batch: dict, task_embedding: torch.Tensor) -> dict:
        """
        Augment observation with task embedding.
        
        Concatenates task embedding to observation.state.
        """
        augmented_batch = {}
        
        for key, value in batch.items():
            if key == 'observation.state':
                # Expand task embedding to match batch size
                if value.dim() == 1:
                    value = value.unsqueeze(0)  # Add batch dim
                
                batch_size = value.shape[0]
                task_embed_expanded = task_embedding.expand(batch_size, -1)
                
                # Concatenate: [state, task_embedding]
                augmented_state = torch.cat([value, task_embed_expanded], dim=-1)
                augmented_batch[key] = augmented_state
            else:
                augmented_batch[key] = value
        
        return augmented_batch
    
    def forward(self, batch: dict, task_name: str = None):
        """
        Forward pass for training.
        
        Args:
            batch: Training batch with observations and actions
            task_name: Task name for this batch
        """
        if task_name is None:
            raise ValueError("task_name must be provided for training")
        
        # Get task embedding
        task_embedding = self.get_task_embedding_simple(task_name)
        
        # Augment observation with task embedding
        augmented_batch = self.augment_observation(batch, task_embedding)
        
        # Forward through base ACT
        return self.act_policy(augmented_batch)
    
    def select_action(self, batch: dict) -> torch.Tensor:
        """
        Select action for inference.
        
        Uses current task set via set_task().
        """
        if self._current_task is None:
            raise ValueError("Task not set. Call set_task() first.")
        
        # Augment observation
        augmented_batch = self.augment_observation(batch, self._current_task_embedding)
        
        # Get action from base policy
        return self.act_policy.select_action(augmented_batch)
    
    def save_pretrained(self, save_path: str):
        """Save model checkpoint."""
        save_path = Path(save_path)
        save_path.mkdir(parents=True, exist_ok=True)
        
        # Save task info
        import json
        task_info = {
            'task_names': self.task_names,
            'task_embed_dim': self.task_embed_dim,
            'original_state_dim': self.original_state_dim,
        }
        with open(save_path / 'task_info.json', 'w') as f:
            json.dump(task_info, f)
        
        # Save model weights
        torch.save(self.state_dict(), save_path / 'model.pt')
        
        # Save base ACT config
        self.act_policy.config.save_pretrained(save_path)
        
        print(f"✅ Model saved to {save_path}")
    
    @classmethod
    def from_pretrained(cls, load_path: str):
        """Load model from checkpoint."""
        load_path = Path(load_path)
        
        import json
        with open(load_path / 'task_info.json', 'r') as f:
            task_info = json.load(f)
        
        # Load base config
        base_config = ACTConfig.from_pretrained(load_path)
        
        # Create model
        model = cls(
            base_config=base_config.__dict__,
            task_names=task_info['task_names'],
            task_embed_dim=task_info['task_embed_dim'],
        )
        
        # Load weights
        model.load_state_dict(torch.load(load_path / 'model.pt'))
        
        print(f"✅ Model loaded from {load_path}")
        return model

print("✅ TaskConditionedACTPolicy defined")

---
## 📊 Create Multi-Task Dataset

Combine datasets from multiple tasks with task labels.

In [ ]:
# ==========================================
# MULTI-TASK DATASET LOADER
# ==========================================

from torch.utils.data import Dataset, DataLoader, ConcatDataset
from lerobot.datasets.lerobot_dataset import LeRobotDataset
import numpy as np

class TaskLabeledDataset(Dataset):
    """
    Wrapper that adds task label to each sample.
    """
    
    def __init__(self, base_dataset: LeRobotDataset, task_name: str):
        self.base_dataset = base_dataset
        self.task_name = task_name
    
    def __len__(self):
        return len(self.base_dataset)
    
    def __getitem__(self, idx):
        sample = self.base_dataset[idx]
        sample['task_name'] = self.task_name
        return sample


def create_multitask_dataloader(
    dataset_repo_ids: dict[str, str],
    batch_size: int = 8,
    shuffle: bool = True,
):
    """
    Create a dataloader that combines multiple task datasets.
    
    Args:
        dataset_repo_ids: Dict of {task_name: repo_id}
        batch_size: Batch size
        shuffle: Whether to shuffle
    
    Returns:
        DataLoader with samples from all tasks
    """
    datasets = []
    
    for task_name, repo_id in dataset_repo_ids.items():
        print(f"📥 Loading dataset: {repo_id}")
        try:
            base_dataset = LeRobotDataset(repo_id)
            labeled_dataset = TaskLabeledDataset(base_dataset, task_name)
            datasets.append(labeled_dataset)
            print(f"   ✅ Loaded {len(base_dataset)} samples")
        except Exception as e:
            print(f"   ⚠️ Failed to load {repo_id}: {e}")
    
    if not datasets:
        raise ValueError("No datasets loaded!")
    
    # Combine all datasets
    combined_dataset = ConcatDataset(datasets)
    print(f"\n📊 Total samples: {len(combined_dataset)}")
    
    dataloader = DataLoader(
        combined_dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=2,
        pin_memory=True,
    )
    
    return dataloader, combined_dataset

print("✅ Multi-task dataset utilities defined")

In [ ]:
# ==========================================
# LOAD MULTI-TASK DATASET
# ==========================================

print(f"\n{'='*70}")
print("📊 Loading Multi-Task Dataset")
print(f"{'='*70}\n")

train_loader, train_dataset = create_multitask_dataloader(
    dataset_repo_ids=DATASET_REPO_IDS,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

print(f"\n✅ DataLoader ready: {len(train_loader)} batches")

---
## 🚀 Training Loop

In [ ]:
# ==========================================
# TRAINING SETUP
# ==========================================

import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Using device: {device}")

# Base ACT config
base_config = {
    'input_shapes': {
        'observation.images.image': [3, 480, 480],
        'observation.state': [4],
    },
    'output_shapes': {
        'action': [4],
    },
    'chunk_size': 100,
    'n_obs_steps': 1,
    'dim_model': 256,
    'n_heads': 8,
    'dim_feedforward': 1024,
    'n_encoder_layers': 4,
    'n_decoder_layers': 7,
    'dropout': 0.1,
}

# Create model
print(f"\n🔧 Creating Task-Conditioned ACT...")
model = TaskConditionedACTPolicy(
    base_config=base_config,
    task_names=TASKS,
    task_embed_dim=TASK_EMBED_DIM,
)
model = model.to(device)

# Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

# Learning rate scheduler
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, 
    T_max=TRAINING_STEPS,
    eta_min=1e-6,
)

print(f"✅ Training setup complete")
print(f"   Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# ==========================================
# TRAINING LOOP
# ==========================================

import wandb
from pathlib import Path

# Initialize W&B
wandb.init(
    project="metaworld-multitask-act",
    name=f"multitask-{len(TASKS)}tasks",
    config={
        'tasks': TASKS,
        'training_steps': TRAINING_STEPS,
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'task_embed_dim': TASK_EMBED_DIM,
    }
)

print(f"\n{'='*70}")
print(f"🚀 Starting Multi-Task Training")
print(f"{'='*70}")
print(f"   Tasks: {TASKS}")
print(f"   Steps: {TRAINING_STEPS:,}")
print(f"{'='*70}\n")

# Training
model.train()
global_step = 0
running_loss = 0.0
save_freq = 25000
log_freq = 100

output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

pbar = tqdm(total=TRAINING_STEPS, desc="Training")

while global_step < TRAINING_STEPS:
    for batch in train_loader:
        if global_step >= TRAINING_STEPS:
            break
        
        # Move batch to device
        batch_device = {}
        task_name = batch.pop('task_name')[0]  # Assume same task in batch
        
        for key, value in batch.items():
            if isinstance(value, torch.Tensor):
                batch_device[key] = value.to(device)
            else:
                batch_device[key] = value
        
        # Forward pass
        optimizer.zero_grad()
        
        try:
            output = model(batch_device, task_name=task_name)
            loss = output['loss']
            
            # Backward pass
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            
            running_loss += loss.item()
            global_step += 1
            pbar.update(1)
            
            # Logging
            if global_step % log_freq == 0:
                avg_loss = running_loss / log_freq
                wandb.log({
                    'loss': avg_loss,
                    'lr': scheduler.get_last_lr()[0],
                    'step': global_step,
                })
                pbar.set_postfix({'loss': f'{avg_loss:.4f}'})
                running_loss = 0.0
            
            # Save checkpoint
            if global_step % save_freq == 0:
                ckpt_path = output_dir / f"checkpoint_{global_step}"
                model.save_pretrained(ckpt_path)
                print(f"\n💾 Checkpoint saved: {ckpt_path}")
        
        except Exception as e:
            print(f"\n⚠️ Error at step {global_step}: {e}")
            continue

pbar.close()

# Save final model
final_path = output_dir / "final_model"
model.save_pretrained(final_path)

print(f"\n{'='*70}")
print("✅ Training Complete!")
print(f"   Final model: {final_path}")
print(f"{'='*70}")

wandb.finish()

---
## 🧪 Evaluation

Evaluate the multi-task model on each task separately.

In [ ]:
# ==========================================
# MULTI-TASK EVALUATION
# ==========================================

import torch
import numpy as np
from lerobot.envs.metaworld import MetaworldEnv
from lerobot.policies.factory import make_pre_post_processors
from lerobot.datasets.lerobot_dataset import LeRobotDatasetMetadata

def evaluate_multitask_policy(
    model: TaskConditionedACTPolicy,
    tasks: list[str],
    dataset_repo_ids: dict[str, str],
    num_episodes_per_task: int = 10,
    max_steps: int = 500,
    device: torch.device = None,
):
    """Evaluate multi-task model on each task."""
    
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    model.eval()
    results = {}
    
    for task_name in tasks:
        print(f"\n{'='*70}")
        print(f"🧪 Evaluating: {task_name}")
        print(f"{'='*70}")
        
        # Set task for inference
        model.set_task(task_name)
        
        # Load dataset stats
        try:
            dataset_meta = LeRobotDatasetMetadata(dataset_repo_ids[task_name])
            dataset_stats = dataset_meta.stats
        except:
            dataset_stats = None
        
        # Create preprocessor/postprocessor
        preprocessor, postprocessor = make_pre_post_processors(
            model.act_policy.config,
            dataset_stats=dataset_stats,
        )
        
        # Create environment
        env = MetaworldEnv(
            task=task_name,
            obs_type="pixels_agent_pos",
            render_mode="rgb_array",
        )
        
        successes = 0
        total_rewards = []
        
        for ep in range(num_episodes_per_task):
            obs, info = env.reset(seed=ep)
            model.reset()
            
            done = False
            steps = 0
            episode_reward = 0
            
            while not done and steps < max_steps:
                # Prepare observation
                batch = {}
                
                image = obs['pixels']
                if isinstance(image, np.ndarray):
                    image = torch.from_numpy(image).float() / 255.0
                if image.shape[-1] == 3:
                    image = image.permute(2, 0, 1)
                batch['observation.images.image'] = image.to(device)
                
                state = obs['agent_pos']
                if isinstance(state, np.ndarray):
                    state = torch.from_numpy(state).float()
                batch['observation.state'] = state.to(device)
                
                # Apply preprocessor
                processed_batch = preprocessor(batch)
                
                # Get action (model already has task set)
                with torch.no_grad():
                    action = model.select_action(processed_batch)
                
                # Apply postprocessor
                action = postprocessor(action)
                
                if isinstance(action, torch.Tensor):
                    action = action.cpu().numpy().squeeze()
                
                obs, reward, terminated, truncated, info = env.step(action)
                done = terminated or truncated
                
                episode_reward += reward
                steps += 1
            
            total_rewards.append(episode_reward)
            
            if info.get('is_success', False):
                successes += 1
                status = "✅"
            else:
                status = "❌"
            
            print(f"   Episode {ep+1}: {status} | Reward: {episode_reward:.2f}")
        
        env.close()
        
        success_rate = 100 * successes / num_episodes_per_task
        avg_reward = np.mean(total_rewards)
        
        results[task_name] = {
            'success_rate': success_rate,
            'successes': successes,
            'avg_reward': avg_reward,
        }
        
        print(f"\n📊 {task_name}: {success_rate:.1f}% ({successes}/{num_episodes_per_task})")
    
    # Print summary
    print(f"\n{'='*70}")
    print("📊 MULTI-TASK EVALUATION SUMMARY")
    print(f"{'='*70}")
    print(f"{'Task':<25} {'Success Rate':>15} {'Avg Reward':>15}")
    print("-" * 70)
    
    for task, res in results.items():
        print(f"{task:<25} {res['success_rate']:>14.1f}% {res['avg_reward']:>15.2f}")
    
    avg_success = np.mean([r['success_rate'] for r in results.values()])
    print("-" * 70)
    print(f"{'AVERAGE':<25} {avg_success:>14.1f}%")
    print(f"{'='*70}")
    
    return results

print("✅ Evaluation function ready")

In [ ]:
# ==========================================
# RUN EVALUATION
# ==========================================

# Load the trained model
model_path = Path(OUTPUT_DIR) / "final_model"

if model_path.exists():
    print(f"📥 Loading model from: {model_path}")
    eval_model = TaskConditionedACTPolicy.from_pretrained(model_path)
    eval_model = eval_model.to(device)
    eval_model.eval()
    
    results = evaluate_multitask_policy(
        model=eval_model,
        tasks=TASKS,
        dataset_repo_ids=DATASET_REPO_IDS,
        num_episodes_per_task=10,
        max_steps=500,
        device=device,
    )
else:
    print(f"⚠️ Model not found at {model_path}")
    print("   Run training first!")

In [ ]:
# ==========================================
# UPLOAD TO HUGGINGFACE
# ==========================================

from huggingface_hub import HfApi

model_path = Path(OUTPUT_DIR) / "final_model"

if model_path.exists():
    POLICY_REPO_ID = f"{HF_USERNAME}/multitask-act-metaworld"
    
    print(f"📤 Uploading to: {POLICY_REPO_ID}")
    
    api = HfApi()
    api.create_repo(repo_id=POLICY_REPO_ID, repo_type="model", exist_ok=True)
    api.upload_folder(
        folder_path=str(model_path),
        repo_id=POLICY_REPO_ID,
        repo_type="model",
    )
    
    print(f"\n✅ Upload complete!")
    print(f"🔗 URL: https://huggingface.co/{POLICY_REPO_ID}")
else:
    print("⚠️ No model to upload")

---
## ✅ Multi-Task Training Complete!

### What you trained:
- **Task-Conditioned ACT** with task token embeddings
- Based on LeX-O approach from your repo
- Model learns to condition behavior on task name

### How it works:
1. Task name → Tokenizer → Token IDs
2. Token IDs → Embedding layer → Task embedding
3. Task embedding concatenated with observation state
4. ACT policy processes augmented observation

### Next steps:
1. Add more tasks by generating more datasets
2. Compare single-task vs multi-task performance
3. Try different task embedding sizes
4. Experiment with tokenized vs simple embeddings

### Adding more tasks:
```python
TASKS = [
    "pick-place-v3",
    "handle-pull-v3",
    "push-v3",  # Add after generating dataset
    "reach-v3",
    "shelf-place-v3",
    "pick-place-wall-v3",
]
```